In [ ]:
import warnings
warnings.simplefilter("ignore")
import plotly.graph_objects as go
import plotly.express as px 
import plotly.offline as pyo
pyo.init_notebook_mode(connected=True)
from sklearn.metrics import precision_recall_curve, auc
from xgboost import XGBClassifier
from sklearn.metrics import recall_score, precision_score
from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import make_scorer
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from category_encoders.count import CountEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
import pandas as pd
import mlflow
import pickle
import logging
import os
import yaml
from contextlib import nullcontext
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

In [ ]:
mlflow_tracking_uri = 'http://localhost:5555' 

# 0. FUNCIONES AUXILIARES.

In [ ]:
def add_estimator_prefix(list_models_grids, prefix='estimator__'):
    return {
        model_name: {
            f"{prefix}{param}": values
            for param, values in params.items()
        }
        for model_name, params in list_models_grids.items()
    }

In [ ]:
def remove_estimator_prefix(param_dict, prefix='estimator__'):
    return {
        key[len(prefix):] if key.startswith(prefix) else key: value
        for key, value in param_dict.items()
    }

# 1. CARGA DE DATA.

In [ ]:
df = pickle.load(open(f'../data/processed/df_cleaned_featured.sav', 'rb'))

# 2. CONFIGURACIONES NECESARIAS.

In [ ]:
scores = {'f1': 'f1',
          'precision': 'precision',
          'recall': 'recall',
          'm_c': make_scorer(matthews_corrcoef)}

In [ ]:
seed = 5000
ratio_balance = 1
k_folds = 3
verbose = 10
test_size = 0.25

In [ ]:
features_names =  ['FLAG_OWN_CAR',
                   'FLAG_OWN_REALTY',
                   'NAME_INCOME_TYPE',
                   'NAME_EDUCATION_TYPE',
                   'NAME_FAMILY_STATUS',
                   'NAME_HOUSING_TYPE',
                   'OCCUPATION_TYPE',
                   'FLAG_WORK_PHONE',
                   'FLAG_PHONE',
                   'FLAG_EMAIL',
                   'AGE',
                   'CNT_CHILDREN',
                   'AMT_INCOME_TOTAL',
                   'WORK_YEARS']
objective_name = 'STATUS'

In [ ]:
features = df[features_names]
list_numeric_names = list(features.select_dtypes(include=['float64', 'int64']).columns)
list_categorical_names = list(features.select_dtypes(include='object').columns)
features = df[list_numeric_names + list_categorical_names]
label = df[objective_name]

In [ ]:
list_models = {
               'HistGradientBoostingClassifier': HistGradientBoostingClassifier(random_state=seed),
               'RandomForestClassifier': RandomForestClassifier(random_state=seed),
               'GradientBoostingClassifier': GradientBoostingClassifier(random_state=seed),
               'LogisticRegression': LogisticRegression(random_state=seed),
               'DecisionTreeClassifier':  DecisionTreeClassifier(random_state=seed),
               'XGBClassifier': XGBClassifier(random_state=seed)
             }

In [ ]:
list_models_grids = {
'HistGradientBoostingClassifier' : {
                                    'max_iter': [100, 200],             
                                    'max_leaf_nodes': [31, 50],         
                                    'min_samples_leaf': [20, 10],      
                                    'learning_rate': [0.1, 0.05]    
                                   },
'RandomForestClassifier'         : {
                                    'n_estimators': [100, 200],  
                                    'max_leaf_nodes': [None, 70], 
                                    'min_samples_leaf': [1, 10], 
                                    'max_features': ['sqrt', 'log2']     
                                   },
'GradientBoostingClassifier'     : {
                                    'n_estimators': [100, 200],
                                    'learning_rate': [0.1, 0.05],
                                    'max_leaf_nodes': [None, 70],
                                    'min_samples_leaf': [1, 10]
                                   },
'LogisticRegression'             : {
                                    'C': [1.0, 0.1],
                                    'penalty': ['l2', 'l1'],
                                    'solver': ['lbfgs', 'liblinear'],
                                    'max_iter': [100, 200]
                                   },
'DecisionTreeClassifier'         : {
                                    'max_depth': [None, 10],
                                    'max_leaf_nodes': [None, 70],
                                    'min_samples_leaf': [1, 10],
                                    'criterion': ['gini', 'entropy']
                                   },
'XGBClassifier'                  : {
                                    'n_estimators': [100, 200],
                                    'learning_rate': [0.1, 0.05],
                                    'max_depth': [6, 10],
                                    'subsample': [1.0, 0.8]
                                   }
}

In [ ]:
prefixed_grids = add_estimator_prefix(list_models_grids)

# 3. SPLIT DE DATA.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features,
                                                    label,
                                                    random_state=seed,
                                                    test_size=test_size,
                                                    stratify=label)

In [ ]:
X_train = X_train[list_numeric_names + list_categorical_names]
X_test = X_test[list_numeric_names + list_categorical_names]

In [ ]:
c_v = StratifiedKFold(n_splits=k_folds,
                        shuffle=True,
                        random_state=seed)

# 4.REGISTRO DE EXPERIMENTOS EN MLFLOW.

In [ ]:
def evaluate_model_with_gridsearch(model=None, 
                                   list_grid=None, 
                                   X_train=None, 
                                   y_train=None,
                                   X_test=None, 
                                   y_test=None,
                                   list_score_names=None,
                                   best_score_name='f1'):
    """
    Evalúa un modelo de clasificación, con la opción de usar GridSearchCV para
    la búsqueda de hiperparámetros, y retorna un diccionario con métricas de rendimiento.

    Si se proporciona un `list_grid`, la función configura un pipeline de preprocesamiento,
    realiza una búsqueda (GridSearch) y así encontrar el mejor modelo. Si no,
    simplemente entrena el modelo proporcionado. Finalmente, evalúa el modelo
    en el conjunto de prueba y calcula diversas métricas.

    Args:
        model (object): Una instancia del modelo de clasificación a evaluar (e.g., LogisticRegression()).
        list_grid (dict, optional): Un diccionario con la malla de hiperparámetros para
                                    GridSearchCV. Si es None, no se realiza la búsqueda.
        X_train (pd.DataFrame or np.ndarray): DataFrame o array con los datos de entrenamiento.
        y_train (pd.Series or np.ndarray): Serie o array con las etiquetas de entrenamiento.
        X_test (pd.DataFrame or np.ndarray): DataFrame o array con los datos de prueba.
        y_test (pd.Series or np.ndarray): Serie o array con las etiquetas de prueba.
        list_score_names (list, optional): Lista de métricas de scoring para usar en Grid
        best_score_name (str, optional): La métrica a optimizar (parámetro 'refit' de  GridSearchCV). Defaults to 'f1'.

    Returns:
        dict: Un diccionario que contiene:
              - 'cm' (np.ndarray): La matriz de confusión.
              - 'f_1' (float): El F1-score.
              - 'recall' (float): El recall score.
              - 'precision' (float): La precision score.
              - 'matthews_corr' (float): El coeficiente de correlación de Matthews.
              - 'accuracy_score' (float): La exactitud (accuracy).
              - 'roc_auc' (float): El área bajo la curva ROC.
              - 'auc_pr' (float): El área bajo la curva Precision-Recall.
              - 'list_precision' (list): Lista de valores de precisión para la curva PR.
              - 'list_recall' (list): Lista de valores de recall para la curva PR.
              - 'model' (object): La mejor instancia del modelo entrenado.
              - 'params' (dict): Los mejores hiperparámetros encontrados.
    """
    if list_grid:
        # CONFIGURACIONES GENERALES. ###########
        numeric_transformer = Pipeline(steps=[('scaler',
                                        StandardScaler())])
        categorical_transformer = Pipeline(steps=[('CountEncoder',
                                            CountEncoder(normalize=True))])
        preprocessor = ColumnTransformer(remainder='passthrough',
                                    transformers=[('numeric',
                                                    numeric_transformer,
                                                    list_numeric_names),
                                                ('categorical',
                                                    categorical_transformer,
                                                    list_categorical_names)])
        train_transform = Pipeline(steps=[('processing',
                                   preprocessor),
                                   ('RandomUnderSampler',
                                    RandomUnderSampler(random_state=seed,
                                                      sampling_strategy=ratio_balance)),
                                  ('estimator',
                                   model)])

        CV_model = GridSearchCV(estimator=train_transform,
                                param_grid=list_grid,
                                cv=c_v,
                                scoring=list_score_names,
                                verbose=verbose,
                                n_jobs=-1,
                                refit=best_score_name)
        CV_model.fit(X_train, y_train)
        best_model = CV_model.best_estimator_
        best_params = CV_model.best_params_
    else:
        model.fit(X_train, y_train)
        best_model = model
        best_params = model.get_params()

    # CÁLCULO DE MÉTRICAS EN EL CONJUNTO DE TESTEO ##########
    y_pred = best_model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    f_1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    matthews_corr = matthews_corrcoef(y_test, y_pred)
    a_s = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred)
    clean_best_params = remove_estimator_prefix(best_params)
    
    # OBTENER PROBABILIDADES ESTIMADAS #########
    probas_pred = best_model.predict_proba(X_test)[:, 1]
    
    # CALCULAR LA CURVA PR Y EL ÁREA BAJO LA CURVA (AUC-PR) ##########
    list_precision, list_recall, _ = precision_recall_curve(y_test, probas_pred)
    auc_pr = auc(list_recall, list_precision)


    return {
        'cm': cm,
        'f_1': f_1,
        'recall': recall,
        'precision': precision,
        'matthews_corr': matthews_corr,
        'accuracy_score': a_s,
        'roc_auc': roc_auc,
        'auc_pr': auc_pr,
        'list_precision': list_precision,
        'list_recall': list_recall,
        'model': best_model,
        'params': clean_best_params
    }

## 4.1 Comienzo de evaluación de modelos.

In [ ]:
if mlflow_tracking_uri:
    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment("Good_Bad_Applicant_Experiment")

In [ ]:
results = {}
pr_curves = []
with mlflow.start_run(run_name="model_comparison") if mlflow_tracking_uri else nullcontext(): 
    for model_name, model in list_models.items():
        logger.info(f"Training {model_name}...")
        with mlflow.start_run(run_name=model_name, nested=True) if mlflow_tracking_uri else nullcontext():
            evaluation = evaluate_model_with_gridsearch(model=model, 
                                                        list_grid=prefixed_grids[model_name], 
                                                        X_train=X_train, 
                                                        y_train=y_train,
                                                        X_test=X_test, 
                                                        y_test=y_test,
                                                        list_score_names=scores,
                                                        best_score_name='f1')
            results[model_name] = evaluation
            
            if mlflow_tracking_uri:
                mlflow.log_params(evaluation['params'])
                mlflow.log_metrics({
                                    'f_1': evaluation['f_1'],
                                    'recall': evaluation['recall'],
                                    'precision': evaluation['precision'],
                                    'matthews_corr': evaluation['matthews_corr'],
                                    'accuracy_score': evaluation['accuracy_score'],
                                    'roc_auc': evaluation['roc_auc']
                                   })
                mlflow.sklearn.log_model(evaluation['model'],
                                         artifact_path=model_name.lower().replace(" ", "_"))
            
            # GUARDAR LA CURVA PR Y ETIQUETA EN LA LISTA ####################
            pr_curves.append((evaluation['list_recall'],
                               evaluation['list_precision'],
                              f"{model_name} (AUC = {evaluation['auc_pr']:.2f})"))
            
            print(f"{model_name}")
            print(f"f1: {evaluation['f_1']}")
            print(f"recall: {evaluation['recall']}")
            print(f"precision: {evaluation['precision']}")
            print(f"matthews_corr: {evaluation['matthews_corr']}")
            print(f"accuracy_score: {evaluation['accuracy_score']}")
            print(f"roc_auc: {evaluation['roc_auc']}")
            print(f"confusion_matrix: {evaluation['cm']}")

## 4.2 Visualización de las curvas PR.

In [ ]:
fig = go.Figure()
colors = px.colors.qualitative.Vivid

# AGREGAR TODAS LAS CURVAS PR
for i, (recall, precision, label) in enumerate(pr_curves):
    fig.add_trace(go.Scatter(x=list(recall),
                             y=list(precision),
                             mode='lines',
                             name=label,
                             line=dict(color=colors[i])))
fig.update_layout(
    title='Curvas PR de Modelos',
    xaxis=dict(title='Recall'),
    yaxis=dict(title='Precision'),
    legend=dict(x=1, y=1),
    showlegend=True,
    template='plotly_white'
)

## 4.3 Nombre del mejor modelo. 

In [ ]:
best_model_name = max(results, key=lambda x: results[x]['f_1'])
best_model_name

In [ ]:
best_model = results[best_model_name]['model']
best_model

In [ ]:
best_f1 = float(results[best_model_name]['f_1'])
best_recall = float(results[best_model_name]['recall'])
best_precision = float(results[best_model_name]['precision'])
best_matthews_corr = float(results[best_model_name]['matthews_corr'])
best_accuracy = float(results[best_model_name]['accuracy_score'])
best_roc = float(results[best_model_name]['roc_auc'])

print(f"🏆 Best Model: {best_model_name}")
print(f"   f1: {best_f1:.4f}")
print(f"   recall: {best_recall:.4f}")
print(f"   precision: {best_precision:.2f}")
print(f"   matthews_corr: {best_accuracy:.2f}")


# 5.GUARDADO DE MEJOR MODELO.

In [ ]:
best_params = {'random_state': seed}
best_params.update(results[best_model_name]['params'])


In [ ]:
best_params

In [ ]:
model_config = {
                'model': {
                    'name': 'good_bad_applicant_model',
                    'best_model': best_model_name,
                    'parameters': best_params,
                    'list_numeric_features': list_numeric_names,
                    'list_categorical_features': list_categorical_names,
                    'target_variable': objective_name,
                    'f1': best_f1,
                    'recall': best_recall
                }
            }

In [ ]:
model_config

In [ ]:
config_path = '../configs/model_config.yaml'
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, 'w') as f:
    yaml.dump(model_config, f)

print(f"se fuar {config_path}")